In [4]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
from torchvision import transforms
import matplotlib.pyplot as plt

In [5]:
transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

train_dataset = torchvision.datasets.GTSRB(
    root='./data',
    split='train',
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.GTSRB(
    root='./data',
    split='test',
    download=True,
    transform=transform
)

In [6]:
train_size = int(0.8*len(train_dataset))
val_size = len(train_dataset) - train_size

train_set, val_set = random_split(train_dataset,[train_size,val_size])

In [10]:
train_loader = DataLoader(train_set,batch_size=128,shuffle=True,num_workers=2)
val_loader = DataLoader(val_set,batch_size=128)
test_loader = DataLoader(test_dataset,batch_size=128)

In [11]:
class CNN(nn.Module):
    def __init__(self, num_classes=43, p=0.3):
        super().__init__()
        self.conv1 = nn.Conv2d(3,32,3,padding=1)
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.conv3 = nn.Conv2d(64,128,3,padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.dropout = nn.Dropout(p)
        self.fc1 = nn.Linear(128*4*4,256)
        self.fc2 = nn.Linear(256,num_classes)

    def forward(self,x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(x.size(0),-1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

def train_epoch(loader):
    model.train()
    total_loss = 0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out,y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss/len(loader)

for epoch in range(16):
    loss = train_epoch(train_loader)
    scheduler.step()
    print(f"Epoch {epoch}, Loss: {loss}")

Epoch 0, Loss: 2.6457534892830306
Epoch 1, Loss: 1.150814905494987
Epoch 2, Loss: 0.5320291959953879
Epoch 3, Loss: 0.2874406859457136
Epoch 4, Loss: 0.19505771619831017
Epoch 5, Loss: 0.13715942826367425
Epoch 6, Loss: 0.09577592650796482
Epoch 7, Loss: 0.07714153193829659
Epoch 8, Loss: 0.05766163098285655
Epoch 9, Loss: 0.05273818544218105
Epoch 10, Loss: 0.026383450569886113
Epoch 11, Loss: 0.021676570571635564
Epoch 12, Loss: 0.018745153943358357
Epoch 13, Loss: 0.019072820726659393
Epoch 14, Loss: 0.01837595452869612
Epoch 15, Loss: 0.016358054447277802


In [13]:
def enable_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

In [14]:
def mc_predict(model, x, T=30):
    model.eval()
    enable_dropout(model)

    probs = []
    with torch.no_grad():
        for _ in range(T):
            out = F.softmax(model(x),dim=1)
            probs.append(out.unsqueeze(0))

    probs = torch.cat(probs,0)  # T x B x K
    mean_probs = probs.mean(0)

    entropy = -torch.sum(mean_probs * torch.log(mean_probs+1e-8),dim=1)
    expected_entropy = -torch.mean(torch.sum(probs*torch.log(probs+1e-8),dim=2),dim=0)
    mutual_info = entropy - expected_entropy

    return mean_probs, entropy, mutual_info

In [15]:
def decision_policy(pred_class, uncertainty, tau=0.5):
    if uncertainty > tau:
        return "EXTEND_RED"
    else:
        return f"ACTION_FOR_CLASS_{pred_class}"